In [1]:
import pandas as pd
import numpy as np
from google.colab import files

In [17]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name)

Saving data_cleaned(Final).xlsx to data_cleaned(Final).xlsx


In [18]:
print("عدد الصفوف والأعمدة:")
print(df.shape)

print("\nأسماء الأعمدة:")
print(df.columns.tolist())

عدد الصفوف والأعمدة:
(240, 32)

أسماء الأعمدة:
['responseId', 'DATE', 'IS', 'IE', 'AA', 'IN', 'F1', 'Q1', 'Q1A', 'primary_reason_Sat', 'secondary_reason_Sat', 'Q2', 'Q2A', 'Q3', 'Q3.1', 'Q3b', 'Q3b.1', 'Q3c', 'Q3c.1', 'Q3d', 'Q3d.1', 'Q5b', 'Q5b.1', 'Q5c', 'Q5c.1', 'Q8', 'Q8.1', 'primary_reason_Rec', 'secondary_reason_Rec', 'start_time_full', 'end_time_full', 'survey_duration_minutes']


In [28]:
# المتغير الهدف: الرضا العام
target_col = "Q1"

# العوامل اللي ممكن تأثر على الرضا
feature_cols = [
    "Q2",    # سهولة التعامل
    "Q3",    # سهولة الحصول على المعلومات
    "Q3b",   # تلبية متطلبات العمل
    "Q3c",   # استقرار الخدمة
    "Q3d",   # الإشعارات المسبقة
    "Q5b",   # جودة إغلاق الطلبات والبلاغات
    "Q5c"    # تجربة معالجة الطلبات والبلاغات
]

# تحويل الأعمدة إلى أرقام
for col in [target_col] + feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 99 missing value
for col in feature_cols:
    df[col] = df[col].replace(99, np.nan)

# إنشاء متغير الرضا
# 1 = راضٍ إذا Q1 = 4 أو 5
# 0 = غير راضٍ أو محايد إذا Q1 = 1 أو 2 أو 3
df["Satisfaction_Status"] = np.where(df[target_col] >= 4, 1, 0)

# تجهيز بيانات النموذج
model_df = df[feature_cols + ["Satisfaction_Status"]].dropna(subset=["Satisfaction_Status"])

print("عدد الصفوف المستخدمة في النموذج:")
print(model_df.shape)

print("\nتوزيع الرضا:")
print(model_df["Satisfaction_Status"].value_counts())

print("\nعدد القيم الناقصة في كل عامل بعد تحويل 99 إلى Missing:")
print(model_df[feature_cols].isna().sum())

عدد الصفوف المستخدمة في النموذج:
(240, 8)

توزيع الرضا:
Satisfaction_Status
1    174
0     66
Name: count, dtype: int64

عدد القيم الناقصة في كل عامل بعد تحويل 99 إلى Missing:
Q2      0
Q3      1
Q3b     1
Q3c     1
Q3d    35
Q5b    22
Q5c    20
dtype: int64


In [22]:
correlations = model_df.corr(numeric_only=True)["Satisfaction_Status"].sort_values(ascending=False)

print("العلاقة بين كل عامل والرضا:")
print(correlations)

العلاقة بين كل عامل والرضا:
Satisfaction_Status    1.000000
Q2                     0.572666
Q3b                    0.546412
Q3c                    0.456971
Q5c                    0.450561
Q3                     0.419399
Q5b                    0.385128
Q3d                    0.267138
Name: Satisfaction_Status, dtype: float64


In [29]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

X = model_df[feature_cols]
y = model_df["Satisfaction_Status"]

# تقسيم البيانات: 80% تدريب، 20% اختبار
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# بناء النموذج
# median مناسب لأن التقييمات من 1 إلى 5
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

model.fit(X_train, y_train)

print("تم تدريب النموذج بنجاح")
print("عدد بيانات التدريب:", X_train.shape[0])
print("عدد بيانات الاختبار:", X_test.shape[0])

تم تدريب النموذج بنجاح
عدد بيانات التدريب: 192
عدد بيانات الاختبار: 48


In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Accuracy:", round(accuracy, 3))
print("Precision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1-score:", round(f1, 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.895
Precision: 0.964
Recall: 0.9
F1-score: 0.931

Confusion Matrix:
[[ 7  1]
 [ 3 27]]

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.88      0.78         8
           1       0.96      0.90      0.93        30

    accuracy                           0.89        38
   macro avg       0.83      0.89      0.85        38
weighted avg       0.91      0.89      0.90        38



In [25]:
# استخراج معاملات النموذج
log_model = model.named_steps["logistic_regression"]

coefficients = pd.DataFrame({
    "Factor": feature_cols,
    "Coefficient": log_model.coef_[0]
})

coefficients["Impact"] = coefficients["Coefficient"].apply(
    lambda x: "Positive impact" if x > 0 else "Negative impact"
)

coefficients = coefficients.sort_values(by="Coefficient", ascending=False)

coefficients

,Factor,Coefficient,Impact
0,Q2,0.958856,Positive impact
3,Q3c,0.900255,Positive impact
2,Q3b,0.868116,Positive impact
1,Q3,0.564563,Positive impact
6,Q5c,0.254936,Positive impact
4,Q3d,-0.133727,Negative impact
5,Q5b,-0.139861,Negative impact


In [26]:
factor_names_ar = {
    "Q2": "سهولة التعامل",
    "Q3": "سهولة الحصول على المعلومات",
    "Q3b": "تلبية متطلبات العمل",
    "Q3c": "استقرار الخدمة",
    "Q3d": "الإشعارات المسبقة",
    "Q5b": "جودة إغلاق الطلبات والبلاغات",
    "Q5c": "تجربة معالجة الطلبات والبلاغات"
}

coefficients["Factor_Arabic"] = coefficients["Factor"].map(factor_names_ar)

coefficients = coefficients[
    ["Factor_Arabic", "Coefficient", "Impact"]
]

coefficients

,Factor_Arabic,Coefficient,Impact
0,سهولة التعامل,0.958856,Positive impact
3,استقرار الخدمة,0.900255,Positive impact
2,تلبية متطلبات العمل,0.868116,Positive impact
1,سهولة الحصول على المعلومات,0.564563,Positive impact
6,تجربة معالجة الطلبات والبلاغات,0.254936,Positive impact
4,الإشعارات المسبقة,-0.133727,Negative impact
5,جودة إغلاق الطلبات والبلاغات,-0.139861,Negative impact


In [31]:
y_pred = model.predict(X_test)

evaluation_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Value": [accuracy, precision, recall, f1]
})

conf_matrix = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    columns=["Predicted_Not_Satisfied", "Predicted_Satisfied"],
    index=["Actual_Not_Satisfied", "Actual_Satisfied"]
)

missing_summary = pd.DataFrame({
    "Factor": feature_cols,
    "Missing_Count": X.isna().sum().values,
    "Missing_Percentage": (X.isna().sum().values / len(X)) * 100
})

missing_summary["Factor_Arabic"] = missing_summary["Factor"].map(factor_names_ar)

output_file = "logistic_regression_results_imputed.xlsx"

with pd.ExcelWriter(output_file) as writer:
    model_df.to_excel(writer, sheet_name="Model_Data", index=False)
    missing_summary.to_excel(writer, sheet_name="Missing_Values", index=False)
    correlations.to_frame("Correlation_With_Satisfaction").to_excel(writer, sheet_name="Correlations")
    coefficients.to_excel(writer, sheet_name="Factor_Importance", index=False)
    evaluation_summary.to_excel(writer, sheet_name="Evaluation", index=False)
    conf_matrix.to_excel(writer, sheet_name="Confusion_Matrix")

files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>